
# The N-Body Simulator

As the name suggests (and as anyone with a cursory interest in physics would know), the objective of the N-body simulation problem is to model the time evolution of a system of N bodies only interacting through the gravitational force given an initial state. Only the 2-body problem can be completely solved - and in the case of very many bodies, such as in the case of modeling galaxy evolution and interactions, this problem becomes quite computationally intensive by a naive approach.

## Formulation

In our case, we will consider the 2D formulation.

We have $n$ point masses in an inertial reference frame, each with mass $m_1,m_2,...m_n$ and position vectors $\vec{r_1},\vec{r_2},...,\vec{r_m}$ only interacting by gravitational force, $$F_{ij} = \frac{Gm_im_j(\vec{r_j}-\vec{r_i})}{||\vec{r_j}-\vec{r_i}||^3}$$
The total gravitational force on mass $i$ is $$F_i = \sum_{j=1, j\neq i}^{n}\frac{Gm_im_j(\vec{r_j}-\vec{r_i})}{||\vec{r_j}-\vec{r_i}||^3}$$

Since the acceleration is second-degree, we should rewrite it in terms of the velocity so we get first-degree for use in RK4.

Our state vector $y$ is then $$y = (\vec{r_1}, \vec{r_2},...,\vec{r_n},\vec{v_1},...,\vec{v_n})$$

For each RK4 step, we use the previous step's velocities to perturb the positions of the current step, then perturb the velocities based on the newly calculated distances & forces.

To avoid extreme wonkiness with particles becoming attracted to each other and then slinging each other at massive speed in away from one another, I've made it so that if two particles are within a certain value, then they will be merged, obeying momentum conservation.
This is accomplished via creating a DSU during force calculation, then merging masses afterwards.

In [ ]:
%matplotlib notebook
from matplotlib import pyplot as plt, animation as anim
import numpy as np
import ipywidgets as widgets
from IPython.display import display

bodies = 5
fig, ax = plt.subplots()
X0 = np.random.randn(bodies) * 10 #stores x-coordinates. Randomly generated at the start.
VX0 = np.random.randn(bodies) * 1 #stores x-velocity
Y0 = np.random.randn(bodies) * 10 #stores y-coordinates. Randomly generated at the start.
VY0 = np.random.randn(bodies) * 1 #stores y-velocity
M = np.full(bodies,500) #stores masses.
C = np.random.random(bodies)
G = 6.67 * 1E-4
DT = 1E-4 #the scaling for time, a debug tool.
gplot = plt.scatter(X0, Y0, c = C, s = 50)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_xlim([-20,20])
ax.set_ylim([-20,20])


"""
def bh_sim(n):
    forces = [0,0]
    return effBodies
"""
def naive_sim(n, X, Y): #calculates forces on an object naively
    forces = [0,0]
    for i in range(bodies):
        if i != n:
            deltaX = X[i] - X[n]
            deltaY = Y[i] - Y[n]
            r_sq = deltaX * deltaX + deltaY * deltaY
            Fg = (G*M[i]*M[n]/r_sq)
            forces[0] += Fg*(deltaX/np.pow(r_sq,0.5))
            forces[1] += Fg*(deltaY/np.pow(r_sq,0.5))
    return forces
def rk4(): #the great integration
    global X0, Y0, VX0,VY0, bodies, DT
    y = np.array([X0, Y0, VX0, VY0])
    FX1 = []
    FY1 = []
    for i in range(bodies):
        forces = naive_sim(i,X0,Y0)
        FX1.append(forces[0])
        FY1.append(forces[1])
    states = np.array([[VX0,VY0,FX1,FY1]])#this array holds all the k's
    for i in range (1,4):
        tf = 0.5
        if i == 3:
            tf = 1
        yn = y + (states[i-1] * tf * DT)
        FXN = []
        FYN = []
        for k in range(bodies):
            forces = naive_sim(k,yn[0],yn[1])
            FXN.append(forces[0])
            FYN.append(forces[1])
        ns = np.array([[yn[2],yn[3],FXN,FYN]])
        states = np.concatenate((states,ns), axis = 0)
    yf = y + (DT/6) * (states[0] + (2*states[1]) + (2*states[2]) + states[3])
    X0 = yf[0]
    Y0 = yf[1]
    VX0 = yf[2]
    VY0 = yf[3]
def update(i):
    global gplot, X0, Y0
    rk4()
    gplot.set_offsets(np.column_stack([X0, Y0]))
    return gplot,

ani = anim.FuncAnimation(fig, update, frames=range(100), interval = 50)
plt.show()
